# Generalization Study: Unseen-Cell, Temporal Extrapolation, & External Zero-Shot Transfer
## TE-Q-Transformer Research Framework

**Scientific Research Question:**  
*"Does the model generalize to strictly unseen cells and forward temporal horizons, and how does a model trained on NASA cylindrical cells behave under zero-shot transfer to prismatic CALCE cells of a different chemistry?"*

---

### Evaluation Protocols:
1. **Unseen-Cell Generalization:** Cells `B0018` (132 cycles, 24°C) and `B0032` (39 cycles, 4°C) held out completely from training and scaler fitting.
2. **Temporal Extrapolation:** Cell `B0053` (44°C) with cycles 0–36 used in training and cycles 37–52 evaluated as forward temporal forecasting.
3. **External Zero-Shot Transfer:** NASA-trained model evaluated on external CALCE CS2 prismatic cells (`CS2_35`, `CS2_36`, `CS2_37`, `CS2_38`) with frozen weights and frozen NASA scaler without any retraining or fine-tuning.

In [ ]:
# ==============================================================================
# 0. SETUP ENVIRONMENT AND REPOSITORY PATHS
# ==============================================================================
import os
import sys
from pathlib import Path

MANUAL_REPO_ROOT = None
REPO_NAME = "TE-Q-Transformer-A-Temperature-Embedded-Quantum-Framework-for-Battery-State-of-Health-Estimation"

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path("/kaggle/working") / REPO_NAME,
    Path("/kaggle/working"),
    Path("/content") / REPO_NAME,
    Path("/content"),
]

if MANUAL_REPO_ROOT and Path(MANUAL_REPO_ROOT).exists():
    REPO_ROOT = Path(MANUAL_REPO_ROOT).resolve()
else:
    REPO_ROOT = next(
        (c.resolve() for c in CANDIDATES if (c / "models" / "proposed" / "te_q_transformer.py").exists() or (c / "datasets" / "NASA" / "processed").exists()),
        Path.cwd().resolve()
    )

print(f"[Setup] REPO_ROOT resolved to: {REPO_ROOT}")
DATA_ROOT = REPO_ROOT / "datasets"
MODEL_ROOT = REPO_ROOT / "models"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml

import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Hardware device: {DEVICE}")

## 1. NASA Ames Dataset Preparation
Loading the NASA dataset and isolating held-out test splits.

In [ ]:
from src.data.nasa_loader import get_nasa_dataloaders

nasa_dir = DATA_ROOT / "NASA" / "processed"
train_loader, test_loaders, scaler = get_nasa_dataloaders(nasa_dir, batch_size=8)

print(f"[Dataset] Train: {len(train_loader.dataset)} cycles")
for cell_id, loader in test_loaders.items():
    print(f"  - Held-out cell {cell_id:<12}: {len(loader.dataset)} cycles")

## 2. In-Domain Evaluation: Unseen Cells & Temporal Extrapolation
Evaluating model predictions across B0018 (unseen 24°C), B0032 (unseen 4°C), and B0053_test (temporal extrapolation 44°C).

In [ ]:
from models.proposed import TEQTransformer, rich_entangler_config
from src.eval.metrics import calculate_metrics, calculate_macro_metrics

def evaluate_nasa_generalization(model: nn.Module, test_loaders: dict, device: torch.device) -> dict:
    model.eval()
    model.to(device)
    per_cell = {}
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            preds, actuals = [], []
            for bx, by in loader:
                bx = bx.to(device)
                preds.append(model(bx).cpu().numpy())
                actuals.append(by.numpy())
            y_pred = np.concatenate(preds)
            y_true = np.concatenate(actuals)
            per_cell[cell_id] = {
                "metrics": calculate_metrics(y_true, y_pred),
                "preds": y_pred,
                "actuals": y_true,
            }
    macro = calculate_macro_metrics([res["metrics"] for res in per_cell.values()])
    return {"per_cell": per_cell, "macro": macro}

# Instantiate model architecture
model = TEQTransformer(rich_entangler_config())
gen_results = evaluate_nasa_generalization(model, test_loaders, DEVICE)

print("\nNASA Generalization Results (Freshly Initialized Model Demonstration):")
for cell_id, cdata in gen_results["per_cell"].items():
    m = cdata["metrics"]
    print(f"  - Cell {cell_id:<12}: RMSE={m['RMSE']:.5f} | MAE={m['MAE']:.5f} | R2={m['R2']:.5f}")

## 3. NASA → CALCE Zero-Shot External Transfer
Evaluating the model directly on external CALCE CS2 prismatic cells (`CS2_35`, `CS2_36`, `CS2_37`, `CS2_38`) with frozen weights and frozen NASA scaler.

In [ ]:
from src.eval.zero_shot import evaluate_zero_shot_calce

calce_dir = DATA_ROOT / "CALCE" / "processed"
if calce_dir.exists():
    calce_res = evaluate_zero_shot_calce(model, calce_dir, scaler, device=DEVICE)
    print("\nCALCE Zero-Shot Transfer Results:")
    for cell_id, cdata in calce_res["per_cell"].items():
        m = cdata["metrics"]
        print(f"  - Cell {cell_id:<10}: RMSE={m['RMSE']:.5f} | MAE={m['MAE']:.5f} | R2={m['R2']:.5f}")
    
    print(f"\n  >> Macro Average : RMSE={calce_res['macro']['RMSE']:.5f} | MAE={calce_res['macro']['MAE']:.5f} | R2={calce_res['macro']['R2']:.5f}")
    print(f"  >> Pooled Overall: RMSE={calce_res['pooled']['RMSE']:.5f} | MAE={calce_res['pooled']['MAE']:.5f} | R2={calce_res['pooled']['R2']:.5f}")
else:
    print(f"CALCE directory not found at {calce_dir}")

## 4. Cross-Domain Prediction Visualization
Visualizing actual vs predicted degradation curves for CALCE cells.

In [ ]:
if calce_dir.exists() and 'calce_res' in locals():
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
    for ax, (cell_id, cdata) in zip(axes, calce_res["per_cell"].items()):
        y_true = cdata["actuals"]
        y_pred = cdata["predictions"]
        cycles = cdata["cycles"]
        ax.plot(cycles, y_true, 'k-', lw=2, label="Actual SOH")
        ax.plot(cycles, y_pred, 'b--', lw=2, label="Zero-Shot Pred")
        m = cdata["metrics"]
        ax.set_title(f"Cell {cell_id} (RMSE: {m['RMSE']:.4f})")
        ax.set_xlabel("Cycle Index")
        ax.set_ylabel("SOH")
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()